In [12]:
# setup
import anthropic
from dotenv import load_dotenv
from anthropic import Anthropic, APIError

import json
import os

# Load variables from the .env file into the system environment
load_dotenv()

MODEL_NAME = "claude-sonnet-4-5"
MAX_ITERATIONS = 8                        # hard ceiling on model<->tool round trips
MAX_TOOL_RETRIES_PER_CALL = 1             # a single tool failing this many times kills the loop
AUTONOMOUS_REFUND_LIMIT_USD = 50.00       # policy boundary, enforced in code, not by the model

In [13]:
# ==========================================================================
# FAKE DATABASE
# ==========================================================================

_ACCOUNTS_DB = {
    "acct_101": {"account_id": "acct_101", "name": "Dana Whitfield", "tier": "standard", "flagged": False},
    "acct_202": {"account_id": "acct_202", "name": "Marcus Ito", "tier": "premium", "flagged": False},
    "acct_303": {"account_id": "acct_303", "name": "Priya Shah", "tier": "standard", "flagged": True},
}

_ORDERS_DB = {
    "ord_9001": {"order_id": "ord_9001", "account_id": "acct_101", "amount_usd": 19.99, "status": "delivered", "issue": "item_missing"},
    "ord_9002": {"order_id": "ord_9002", "account_id": "acct_202", "amount_usd": 249.00, "status": "delivered", "issue": "not_as_described"},
    "ord_9003": {"order_id": "ord_9003", "account_id": "acct_303", "amount_usd": 34.50, "status": "delivered", "issue": "duplicate_charge"},
}

testOrder = _ORDERS_DB.get("ord_9001") #["ord_9001"]
print(f"Test Order: {testOrder}")

Test Order: {'order_id': 'ord_9001', 'account_id': 'acct_101', 'amount_usd': 19.99, 'status': 'delivered', 'issue': 'item_missing'}


In [14]:
# ==========================================================================
# TOOLS
# These are the hands of the agent - the model can never touch your
# database directly, it can only ASK you to run one of these functions by
# name. Each one returns a dictionary result, or raises ValueError on an expected,
# recoverable failure (bad id, bad amount, etc).
# ==========================================================================

def tool_get_account(account_id):
    """Read-only lookup. Model uses this to see who it's talking to."""
    account = _ACCOUNTS_DB.get(account_id)
    if account is None:
        raise ValueError(f"No account found with id '{account_id}'")
    return account
 
 
def tool_get_order(order_id):
    """Read-only lookup. Model uses this to see what the ticket is about."""
    order = _ORDERS_DB.get(order_id)
    if order is None:
        raise ValueError(f"No order found with id '{order_id}'")
    return order
 
 
def tool_issue_refund(order_id, amount_usd, reason):
    """
    The one HIGH RISK action tool. Note what this function does NOT do:
    it does not check the autonomous refund limit. That check happens one
    layer up, in dispatch_tool(), BEFORE this function is ever called. By
    the time we're inside this function, the amount has already been
    approved as safe to execute autonomously.
    """
    order = _ORDERS_DB.get(order_id)
    if order is None:
        raise ValueError(f"Cannot refund unknown order '{order_id}'")
    if amount_usd <= 0:
        raise ValueError("Refund amount must be positive")
    if amount_usd > order["amount_usd"]:
        raise ValueError(f"Refund amount {amount_usd} exceeds order total {order['amount_usd']}")
 
    return {"status": "refund_issued", "order_id": order_id, "amount_usd": amount_usd, "reason": reason}
 
 
def tool_escalate_to_human(reason, recommended_action):
    """
    Not a real action -- just a signal. Calling this doesn't DO anything
    to a database; it tells the agent loop "stop here, a human needs to look at
    this." The loop watches for this specific tool name below.
    """
    return {"status": "escalated", "reason": reason, "recommended_action": recommended_action}
 
 
def tool_finish(summary):
    """
    Also just a signal, like escalate_to_human. This is how the model tells
    us "I'm done, and I handled it myself" as opposed to just going
    quiet, which we'd treat as a natural (but less explicit) completion.
    """
    return {"status": "finished", "summary": summary}
 
 
# A lookup table mapping the tool name (a string, as sent by the model) to
# the actual Python function that implements it. This lets dispatch_tool()
# find the right function with a single dictionary lookup instead of a long
# if/elif chain.
TOOL_FUNCTIONS = {
    "get_account": tool_get_account,
    "get_order": tool_get_order,
    "issue_refund": tool_issue_refund,
    "escalate_to_human": tool_escalate_to_human,
    "finish": tool_finish,
}

# account = tool_get_account("acct_101")
# print(f"Account: {account}")

In [15]:
# --------------------------------------------------------------------------
# Tool schemas: https://platform.claude.com/docs/en/agents-and-tools/tool-use/define-tools
# This is the "menu" we hand to the model. It tells Claude what tools
# exist and what arguments each one takes. The model reads this and decides
# which tool (if any) to call; it never sees our Python code.
# --------------------------------------------------------------------------

TOOLS = [
    {
        "name": "get_account",
        "description": "Look up a customer account by id. Read-only.",
        "input_schema": {
            "type": "object",
            "properties": {"account_id": {"type": "string"}},
            "required": ["account_id"],
        },
    },
    {
        "name": "get_order",
        "description": "Look up an order by id. Read-only.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "issue_refund",
        "description": "Issue a refund for an order. HIGH RISK -- refunds above the "
                        "autonomous limit will be blocked and must be escalated instead.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "amount_usd": {"type": "number"},
                "reason": {"type": "string"},
            },
            "required": ["order_id", "amount_usd", "reason"],
        },
    },
    {
        "name": "escalate_to_human",
        "description": "Hand this ticket off to a human instead of acting autonomously.",
        "input_schema": {
            "type": "object",
            "properties": {
                "reason": {"type": "string"},
                "recommended_action": {"type": "string"},
            },
            "required": ["reason", "recommended_action"],
        },
    },
    {
        "name": "finish",
        "description": "Call this once you've fully resolved the ticket yourself.",
        "input_schema": {
            "type": "object",
            "properties": {"summary": {"type": "string"}},
            "required": ["summary"],
        },
    },
]
 
SYSTEM_PROMPT = f"""You are an autonomous SaaS support agent working for a large retail company.

You may issue refunds up to ${AUTONOMOUS_REFUND_LIMIT_USD:.2f} on your own.
Anything larger must be escalated -- call escalate_to_human instead of
attempting it. When you've fully resolved a ticket yourself, call finish
with a short summary."""

In [16]:
# ==========================================================================
# DISPATCH: the one place a tool name (a string) turns into an actual
# function call. This is also where the safety policy lives.
# ==========================================================================
def dispatch_tool(name, tool_input):
    """
    Runs a single tool call and returns (result_dict, was_error).
 
    Two jobs happen here, in this order:
      1. POLICY CHECK -- before running anything, ask "is this action even
         allowed?" A refund request over the limit gets blocked right here,
         before tool_issue_refund() is ever called. The model cannot argue
         its way past this -- it's a plain if-statement, not a suggestion.
      2. EXECUTE -- look up the right function by name and call it, catching
         any expected error so a bad tool call becomes a message we send
         back to the model instead of a crash that kills the program.
    """
 
    # --- 1. Policy check (only relevant for issue_refund) ---
    if name == "issue_refund":
        # get the amount from the tool input, defaulting to 0 if not present
        amount = tool_input.get("amount_usd", 0)
        if amount > AUTONOMOUS_REFUND_LIMIT_USD:
            print(f"Policy block: ${amount:.2f} exceeds the ${AUTONOMOUS_REFUND_LIMIT_USD:.2f} autonomous limit.")
            return (
                {"status": "blocked_by_policy",
                 "reason": f"${amount:.2f} exceeds the ${AUTONOMOUS_REFUND_LIMIT_USD:.2f} "
                           f"autonomous limit. Escalate this instead."},
                True,  # this counts as an error result
            )
 
    # --- 2. Look up and execute the tool ---
    func = TOOL_FUNCTIONS.get(name)
    if func is None:
        # The model asked for a tool that doesn't exist. Shouldn't normally
        # happen, but we handle it instead of crashing.
        return ({"error": f"Unknown tool '{name}'"}, True)
 

    try:
        # **tool_input unpacks the dict into keyword arguments, e.g.
        # {"order_id": "ord_9001"} becomes func(order_id="ord_9001")
        result = func(**tool_input)
        return (result, False)
    except ValueError as e:
        # An expected, "normal" failure raised by the tool itself
        # (bad id, bad amount, etc).
        return ({"error": str(e)}, True)
    except TypeError as e:
        # The model sent arguments that don't match what the function
        # expects (missing/misspelled keys). Still not a crash -- we tell
        # the model what went wrong instead.
        return ({"error": f"Bad arguments for {name}: {e}"}, True)

In [21]:
# ==========================================================================
# THE LOOP
# The core send -> read -> act -> repeat cycle. Everything above this is
# just setup; this function is where they all get used together.
# ==========================================================================
 
def run_loop(client, ticket_text):
    """
    Runs one support ticket at a time through the agent loop until it reaches a
    terminal outcome. Returns (outcome_string, trace_list, summary_string).
 
    WHY THIS CAN'T RUN FOREVER:
    Every single pass through the `while True` loop below either (a) hits
    a `return` statement and exits immediately, or (b) falls through to the
    bottom, where `iteration` gets incremented by exactly 1. Since the very
    first check in the loop compares `iteration` to MAX_ITERATIONS and
    returns if we've hit the cap, the loop can run at most MAX_ITERATIONS
    times -- full stop, regardless of what the model or tools do.
    """
    # `messages` is the running conversation history we send back to the
    # model every turn -- it has to see everything that happened so far to
    # decide what to do next.
    messages = [{"role": "user", "content": ticket_text}]
 
    # Just a running log of what happened, turn by turn, for the printout.
    trace = []
 
    # Tracks how many times each exact (tool, arguments) pair has failed,
    # so a broken tool can't be retried forever.
    failure_counts = {}
 
    iteration = 0
    while True:
        # --- Hard stop #1: iteration cap, checked FIRST, every pass ---
        if iteration >= MAX_ITERATIONS:
            trace.append(f"Hit max iterations -- stopping. Iteration count = {iteration}")
            return "MAX_ITERATIONS_HIT", trace, "Loop terminated: too many iterations."
 
        iteration += 1
 
        # --- Ask the model what to do next ---
        response = client.messages.create(
            model=MODEL_NAME, # claude-sonnet-4-5 
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )
 
        # The API requires we echo the model's own turn back into the
        # conversation history before continuing.
        messages.append({"role": "assistant", "content": response.content})
 
        # Log any plain text the model said (its reasoning, a draft reply, etc).
        for block in response.content:
            if block.type == "text" and block.text.strip():
                trace.append(f"Iteration Counter: {iteration}\n")
                trace.append(f"Claude said: {block.text.strip()}")
 
        # --- Case 1: model is done, no tool calls this turn ---
        if response.stop_reason == "end_turn":
            trace.append("Model stopped naturally (no tool call).")
            last_text = next((b.text for b in reversed(response.content) if b.type == "text"), "")
            return "RESOLVED_AUTONOMOUSLY", trace, last_text
 
        # --- Case 2: something other than a normal tool call happened ---
        # (e.g. ran out of tokens mid-response) -- treat as a controlled
        # failure rather than silently continuing.
        if response.stop_reason != "tool_use":
            trace.append(f"Unexpected stop_reason '{response.stop_reason}' -- stopping.")
            return "FAILED", trace, "Loop terminated: unexpected stop reason."
 
        # --- Case 3: the model wants to call one or more tools ---
        tool_results = []       # what we'll send back to the model next turn
        finished = False
        escalated = False
        summary = ""
 
        for block in response.content:
            if block.type != "tool_use":
                continue  # skip text blocks, we already logged those above
 
            trace.append(f"Tool call: {block.name}({block.input})\n\n")
 
            result, was_error = dispatch_tool(block.name, block.input)
 
            if was_error:
                # Track failures per exact (tool, args) combo -- so a
                # different call to the same tool doesn't get blamed.
                key = f"{block.name}:{json.dumps(block.input, sort_keys=True)}"
                failure_counts[key] = failure_counts.get(key, 0) + 1
                trace.append(f"  -> ERROR: {result}")
 
                if failure_counts[key] > MAX_TOOL_RETRIES_PER_CALL:
                    trace.append(f"'{block.name}' failed too many times -- stopping instead of spinning.")
                    return "FAILED", trace, f"Loop terminated: {block.name} kept failing."
            else:
                trace.append(f"  -> OK: {result}")
 
            # Every tool call needs a matching tool_result, whether it
            # succeeded or failed -- the model needs to see the outcome of
            # EVERY call it made, not just the successful ones.
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,   # links this result back to the specific call
                "content": json.dumps(result),
                "is_error": was_error,
            })
 
            # Watch for our two "signal" tools -- these don't get detected
            # via stop_reason, we have to check which tool was actually
            # called.
            if block.name == "finish" and not was_error:
                finished = True
                summary = result.get("summary", "")
            if block.name == "escalate_to_human" and not was_error:
                escalated = True
                summary = result.get("reason", "")
 
        # Send all the tool results back as a single new message -- this is
        # the format the API expects: tool results always arrive as a
        # "user" role message containing tool_result blocks.
        messages.append({"role": "user", "content": tool_results})
 
        if finished:
            trace.append("Model called finish() -- done.")
            return "RESOLVED_AUTONOMOUSLY", trace, summary
 
        if escalated:
            trace.append("Model called escalate_to_human() -- handing off.")
            return "ESCALATED", trace, summary
 
        # Neither finish nor escalate was called -- loop back around with
        # the tool results now in `messages`, so the model can see what
        # happened and decide what to do next.
 

In [ ]:
# ==========================================================================
# DEMO
# Run a few sample tickets through the loop and print what happened.
# ==========================================================================
 
def print_result(label, ticket_text, outcome, trace, summary):
    print("\n")
    print("=" * 70)
    print(f"TICKET: {label}")
    print(f"  {ticket_text}")
    print("-" * 70)
    for line in trace:
        print(f"  {line}")
    print("-" * 70)
    print(f"OUTCOME: {outcome}")
    if summary:
        print(f"SUMMARY: {summary}")
    print()
 
 
def main():

# Retrieve the variables using os.getenv()
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        print("Set ANTHROPIC_API_KEY first.")
        return
    else:
        print(f"ANTHROPIC_API_KEY: {api_key}")
 
    client = Anthropic(api_key=api_key)
 
    tickets = [
        ("Small refund, in-policy",
         "My order ord_9001 (account acct_101) was missing an item. "
         "I paid $19.99. Can I get a refund?"),
 
        ("Large refund, needs escalation",
         "Order ord_9002 (account acct_202) was totally wrong. "
         "I want my full $249.00 back."),
 
        ("Bad order id, triggers failure path",
         "I was double charged for order ord_9999 on account acct_303. "
         "Please refund it."),
    ]
 
    for label, text in tickets:
        outcome, trace, summary = run_loop(client, text)
        print_result(label, text, outcome, trace, summary)
 
 
if __name__ == "__main__":
    main()